# A Simulation of the BB84 Quantum Key Distribution (QKD) Protocol


## 1. Setup & Utilities

- Import necessary packages

In [ ]:
import qiskit
import math
import numpy as np
from numpy.typing import NDArray
import pickle
import random
from typing import List, Tuple

from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit.primitives import StatevectorSampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit.primitives import BackendSamplerV2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

- Save credentials locally. This is one-time setup, does not need to re-run every times we build the notebook.

In [ ]:
# Configuration
import os
USE_REAL_HARDWARE = True
RT_SERVICE_INIT = False
SAVE_DATA = True
token = os.environ.get("IBM_PW", None)
if token is None:
    USE_REAL_HARDWARE = False

In [ ]:
if USE_REAL_HARDWARE and not RT_SERVICE_INIT:
    # Stores credentials locally one-time setup
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        overwrite=True,
        set_as_default=True
    )
    RT_SERVICE_INIT = True


Connect to IBM Quantum hardware and choose the hardware

In [ ]:
service = None
hw_backend = None
sim_backend = AerSimulator()
if USE_REAL_HARDWARE:
    # Connect to IBM Quantum hardware
    service = QiskitRuntimeService()

    hw_backend = service.least_busy(
        operational=True,
        simulator=False,
        min_num_qubits=127
    )

print("Backend:", getattr(hw_backend, "name", None))
print("Sim Backend:", getattr(sim_backend, "name", None))

In [ ]:
# Inspired by assignment 11 example and https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models#noise-example-1-basic-bit-flip-error-noise-model
noise_model = NoiseModel()
error = depolarizing_error(0.05, 1)
error2 = error.tensor(error)
p_msmt_error = 0.1 
error_meas = pauli_error([("X", p_msmt_error), ("I", 1 - p_msmt_error)])
noise_model.add_all_qubit_quantum_error(error, ["u1", "u2", "u3", "h"])
noise_model.add_all_qubit_quantum_error(error2, ["cx"])
noise_model.add_all_qubit_quantum_error(error_meas, "measure")
noisy_sim_backend = AerSimulator(noise_model=noise_model)

- Helpers for experiments

In [ ]:
def get_rn_generator(seed=None):
    """
    This helper create a random number generator

    Params:
        seed: Random seed for reproducibility. If None, results vary each run.

    Returns:
         A random number generator
    """
    return np.random.default_rng(seed)

def sample_bits(bit_num, rng):
    """Sample an array of random 0|1 bits."""
    return rng.integers(0, 2, size=bit_num)

def encode_bb84_states(qc, bits, bases):
    """
    Encode BB84 states onto the qubits in-place.

    Convention:
    bit=0, basis=0 -> |0>
    bit=1, basis=0 -> |1>
    bit=0, basis=1 -> |+>
    bit=1, basis=1 -> |->
    """
    bit_num = len(bits)

    for n in range(bit_num):
        if bits[n] == 0:
            if bases[n] == 1:
                qc.h(n)
        if bits[n] == 1:
            if bases[n] == 0:
                qc.x(n)
            if bases[n] == 1:
                qc.x(n)
                qc.h(n)

def measure_bb84(qc, bases):
    """
    Measure qubits in BB84 bases in-place.

    basis=0 -> Z measurement
    basis=1 -> X measurement via H then Z measurement
    """
    bit_num = len(bases)

    for n in range(bit_num):
        if bases[n] == 1:
            qc.h(n)
        qc.measure(n, n)

def run_statevector_sampler(qc, shots=1):
    """
    Run a circuit locally with StatevectorSampler.
    """
    sampler = StatevectorSampler()
    return sampler.run([qc], shots=shots).result()


def extract_counts(result):
    """
    Extract both string and integer counts from result.
    """
    counts = result[0].data.c.get_counts()
    countsint = result[0].data.c.get_int_counts()
    return counts, countsint


def counts_to_bbits(counts, bit_num):
    """
    Convert the first measured bitstring in counts to Bob's bit array.

    Assumes shots=1 or that the first key is the one you want to inspect.
    Reverses bit order to match qubit indexing.
    """
    keys = counts.keys()
    key = list(keys)[0]
    bmeas = list(key)

    bmeas_ints = []
    for n in range(bit_num):
        bmeas_ints.append(int(bmeas[n]))

    bbits = bmeas_ints[::-1]
    return bbits


def sift_key(sender_bits, sender_bases, receiver_bases, receiver_bits):
    """
    Keep only positions where sender and receiver bases match.
    """
    sender_sifted = []
    receiver_sifted = []

    for n in range(len(sender_bits)):
        if sender_bases[n] == receiver_bases[n]:
            sender_sifted.append(int(sender_bits[n]))
            receiver_sifted.append(int(receiver_bits[n]))

    return sender_sifted, receiver_sifted


def key_statistics(sender_sifted, receiver_sifted):
    """
    Compute match count, fidelity, and loss for sifted keys.
    """
    if len(sender_sifted) == 0:
        return {
            "match_count": 0,
            "sifted_length": 0,
            "fidelity": None,
            "loss": None,
        }

    match_count = 0
    for n in range(len(sender_sifted)):
        if sender_sifted[n] == receiver_sifted[n]:
            match_count += 1

    fidelity = match_count / len(sender_sifted)
    loss = 1 - fidelity

    return {
        "match_count": match_count,
        "sifted_length": len(sender_sifted),
        "fidelity": fidelity,
        "loss": loss,
    }


def print_protocol_summary(abits, abase, bbase, bbits, agoodbits, bgoodbits, stats, ebits=None, ebase=None):
    """
    Print a compact summary for one BB84-like run.
    """
    print("Alice's bits are ", list(abits))
    print("Alice's bases are ", list(abase))
    if ebase is not None and ebits is not None:
        print("Eve base:   ", ebase)
        print("Eve bits:   ", ebits)
    print("Bob's bases are   ", list(bbase))
    print("Bob's measured bits:", bbits)
    print("Alice sifted key:", agoodbits)
    print("Bob sifted key:  ", bgoodbits)

    if stats["sifted_length"] > 0:
        print("match_count =", stats["match_count"])
        print("fidelity    =", stats["fidelity"])
        print("loss        =", stats["loss"])
    else:
        print("No sifted bits were produced.")
    

In [ ]:
def simulate_bb84(num_try, losses, fidelities, backend=hw_backend, bit_num: int = 20, seed: int = 123, sampler = None):
    # Setup
    rng = get_rn_generator(seed)
    if sampler is None:
        if service is not None:
            sampler = Sampler(mode=backend)
        else:
            sampler = StatevectorSampler()

    # Alice selects random bits and bases
    abits = sample_bits(bit_num, rng)
    abase = sample_bits(bit_num, rng)
    
    # 20 qubits, 20 classical bits for measurement results
    qc = QuantumCircuit(bit_num, bit_num)
    
    # Encoding: Alice prepares qubits
    encode_bb84_states(qc, abits, abase)
    qc.barrier()
    
    # Bob measures in his chosen bases
    bbase = sample_bits(bit_num, rng)
    measure_bb84(qc, bbase)

    if num_try == 0:
        display(qc.draw("mpl"))
    
    # Transpile for the backend
    pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
    qc_isa = pm.run(qc)

    # Execute the job
    job = sampler.run([qc_isa], shots=1)

    result = job.result()
    
    # Post-process
    counts, countsint = extract_counts(result)
    bbits = counts_to_bbits(counts, bit_num)
    
    agoodbits, bgoodbits = sift_key(abits, abase, bbase, bbits)
    stats = key_statistics(agoodbits, bgoodbits)
    
    print_protocol_summary(abits, abase, bbase, bbits, agoodbits, bgoodbits, stats)

    if stats["sifted_length"] > 0:
        fidelities.append(stats["fidelity"])
        losses.append(stats["loss"])
    return {"result": result, "abits": abits, "abase": abase, "bbits": bbits, "bbase": bbase, "agoodbits": agoodbits, "bgoodbits": bgoodbits, "stats": stats}

## 2. Experiment 1: Ideal BB84 QKD Simulate - No Noise, No Eve
This experiment simulate how Alice sends qubits and Bob measures them using random bases to generate a shared secret key.
- Alice encodes random bits into qubits using random bases (Z or X)
- Bob measures using random bases
- Later, they compare bases and keep only matching ones

BB84 with a noise-free simulator

In [ ]:
# Entry point
trial_num = 5
fidelities = []
losses = []
record = []

for trial in range(trial_num):
    print("Try ", trial)
    record.append(simulate_bb84(trial, losses, fidelities, backend=sim_backend))
    
print("Average fidelity =", sum(fidelities) / len(fidelities))
print("Average loss     =", sum(losses) / len(losses))

if SAVE_DATA:
    with open("experiment1.pkl", "wb") as fid:
        pickle.dump(record, fid)

In [ ]:
# Entry point
trial_num = 5
fidelities = []
losses = []
record = []

for trial in range(trial_num):
    print("Try ", trial)
    record.append(simulate_bb84(trial, losses, fidelities, backend=noisy_sim_backend))
    
print("Average fidelity =", sum(fidelities) / len(fidelities))
print("Average loss     =", sum(losses) / len(losses))

if SAVE_DATA:
    with open("experiment1_noisy.pkl", "wb") as fid:
        pickle.dump(record, fid)

## 3. Experiment 2: BB84 QKD on IBM hardware - Hardware Noise
BB84 on IBM hardware

In [ ]:
# Entry point
trial_num = 10
fidelities = []
losses = []
record = []

for trial in range(trial_num):
    print("Try ", trial)
    record.append(simulate_bb84(trial, losses, fidelities, backend=hw_backend))
    
print("Average fidelity =", sum(fidelities) / len(fidelities))
print("Average loss     =", sum(losses) / len(losses))

if SAVE_DATA:
    with open("experiment2.pkl", "wb") as fid:
        pickle.dump(record, fid)

## 4. Experiment 3: BB84 QKD with Eavesdropper on IBM Hardware (Intercept-Resend, Eve + Noise)
First attack model:
- Eve random basis and measure qubit
- Eve resend it to Bob
- QBER raises ~25% theoretical

In [ ]:
def eavesdropper_experiment(num_try, losses, fidelities, backend=hw_backend, bit_num: int = 20, seed: int = 123, sampler = None):
    # Setup
    rng = get_rn_generator(seed)
    if sampler is None:
        if service is not None:
            sampler = Sampler(mode=backend)
        else:
            sampler = StatevectorSampler()

    # Alice selects random bits and bases
    abits = sample_bits(bit_num, rng)
    abase = sample_bits(bit_num, rng)
    
    # Part 1: Alice -> Eve
    qc_eve = QuantumCircuit(bit_num, bit_num)
    
    # Alice prepares qubits
    encode_bb84_states(qc_eve, abits, abase)
    qc_eve.barrier()
    
    # Eve measures in her chosen bases
    ebase = sample_bits(bit_num, rng)
    measure_bb84(qc_eve, ebase)

    # Transpile for the backend
    pm = generate_preset_pass_manager(
        target=backend.target,
        optimization_level=3
    )
    qc_eve_isa = pm.run(qc_eve)
    
    # Execute Eve's interception on hardware
    job_eve = sampler.run([qc_eve_isa], shots=1)
    print("Eve Job ID:", job_eve.job_id())
    result_eve = job_eve.result()
    counts_eve, countsint_eve = extract_counts(result_eve)
    # Post-processing
    ebits = counts_to_bbits(counts_eve, bit_num)
    print("Eve's bits:\n", ebits)
    
    
    # Part 2: Eve -> Bob
    # Eve construct guess states to send on to Bob
    qc_bob = QuantumCircuit(bit_num, bit_num)
    # Eve encodes using her measured bits in her chosen bases
    encode_bb84_states(qc_bob, ebits, ebase)
    qc_bob.barrier()
    
    # Bob measures in his chosen bases
    bbase = sample_bits(bit_num, rng)
    measure_bb84(qc_bob, bbase)
    
    # Transpile for the backend
    qc_bob_isa = pm.run(qc_bob)
    # Execute Bob's measurement on hardware
    job_bob = sampler.run([qc_bob_isa], shots=1)
    print("Bob Job ID:", job_bob.job_id())
    result_bob = job_bob.result()
    # Post-process Bob's measurement
    counts_bob, countsint_bob = extract_counts(result_bob)
    bbits = counts_to_bbits(counts_bob, bit_num)
    
    # Compare Alice's and Bob's bits
    agoodbits, bgoodbits = sift_key(abits, abase, bbase, bbits)
    stats = key_statistics(agoodbits, bgoodbits)
    print_protocol_summary(abits, abase, bbase, bbits, agoodbits, bgoodbits, stats, ebits, ebase)
    
    if stats["sifted_length"] > 0:
        fidelities.append(stats["fidelity"])
        losses.append(stats["loss"])
    return {"abits": abits, "abase": abase, "bbits": bbits, "bbase": bbase, "ebits": ebits, "ebase": ebase, "bob_result": result_bob, "eve_result": result_eve, "agoodbits": agoodbits, "bgoodbits": bgoodbits, "stats": stats}

In [ ]:
# Entry point
trial_num = 5
fidelities = []
losses = []
record = []

for trial in range(trial_num):
    print("Try ", trial)
    record.append(eavesdropper_experiment(trial, losses, fidelities, backend=sim_backend))
    
print("Average fidelity =", sum(fidelities) / len(fidelities))
print("Average loss     =", sum(losses) / len(losses))

if SAVE_DATA:
    with open("experiment3_sim.pkl", "wb") as fid:
        pickle.dump(record, fid)

In [ ]:
# Entry point
trial_num = 5
fidelities = []
losses = []
record = []

for trial in range(trial_num):
    print("Try ", trial)
    record.append(eavesdropper_experiment(trial, losses, fidelities, backend=noisy_sim_backend))
    
print("Average fidelity =", sum(fidelities) / len(fidelities))
print("Average loss     =", sum(losses) / len(losses))

if SAVE_DATA:
    with open("experiment3_noisy_sim.pkl", "wb") as fid:
        pickle.dump(record, fid)

In [ ]:
# Entry point
trial_num = 5
fidelities = []
losses = []
record = []

for trial in range(trial_num):
    print("Try ", trial)
    record.append(eavesdropper_experiment(trial, losses, fidelities, backend=hw_backend))
    
print("Average fidelity =", sum(fidelities) / len(fidelities))
print("Average loss     =", sum(losses) / len(losses))

if SAVE_DATA:
    with open("experiment3_hw_backend.pkl", "wb") as fid:
        pickle.dump(record, fid)

## 5. Experiment 4: Error correction
- Classical post-processing
- parity check
- Show QBER decreases after correction

In [ ]:
def bit_error_rate(
    input_bits: NDArray[np.int64], output_bits: NDArray[np.int64]
) -> NDArray[np.int64]:
    """
    Computes the quantum bit error rate between two sets of bits
    """

    if np.size(input_bits) != np.size(output_bits):
        raise ValueError("input_bits and output_bits must be of the same size")
    if isinstance(input_bits, list):
        input_bits = np.array(input_bits)
    if isinstance(output_bits, list):
        output_bits = np.array(output_bits)

    if np.size(input_bits) == 0:
        return np.nan
    else:
        return np.sum(input_bits.flatten() != output_bits.flatten()) / np.size(
            input_bits
        )


def parity(bits: NDArray[np.int64]) -> np.int64:
    """
    Computes the parity of a set of bits
    """

    return int(np.bitwise_xor.reduce(bits))


def chunks(arr: NDArray[np.number], n: np.int64) -> List[NDArray[np.number]]:
    """
    Splits a N-element array into separate arrays each of size n where each entry
    is the corresponding index in the original array.

    Example: chunks(np.array([1, 2, 3, 4]), 2) -> [np.array([0, 1]), np.array([2, 3])]
    """

    return np.split(np.arange(arr.size), np.arange(n, len(arr), n))


def binary_search_parity_error(
    arr1: NDArray[np.int64], arr2: NDArray[np.int64], idx: NDArray[np.int64]
) -> Tuple[int, int]:
    """
    Performs a binary search on two arrays to locate a parity mismatch.
    """

    leakage = 0
    while idx.size > 1:
        mid = idx.size // 2  # Midpoint index
        left_idx = idx[:mid]  # Left side of array

        leakage += 1
        # Compare parity of blocks
        if parity(arr1[left_idx]) != parity(arr2[left_idx]):
            idx = left_idx
        else:
            idx = idx[mid:]

    return int(idx[0]), leakage


def cascade_protocol(
    alice_key: NDArray[np.int64],
    bob_key: NDArray[np.int64],
    chunk_size: int,
    idx_perm=None,
) -> Tuple[NDArray[np.int64], NDArray[np.int64], NDArray[np.int64]]:
    """
    Implementation of Cascade Protocol
    """

    if idx_perm is None:
        idx_perm = np.arange(alice_key.size)

    # Re-organize bits according to input permutation
    print(idx_perm, type(idx_perm))
    alice_perm = alice_key[idx_perm]
    bob_perm = bob_key[idx_perm]

    idx_err = np.array([], dtype=np.int64)
    idx_blocks = chunks(np.arange(alice_key.size, dtype=np.int64), chunk_size)
    eve_information = 0
    for idx in idx_blocks:

        # Compute parity of sub-set of bits
        alice_parity = parity(alice_perm[idx])
        bob_parity = parity(bob_perm[idx])

        eve_information += 1
        if alice_parity != bob_parity:
            # Find the error and correct the parity
            err_loc, leakage = binary_search_parity_error(alice_perm, bob_perm, idx)
            eve_information += leakage
            bob_perm[err_loc] ^= 1
            idx_err = np.append(idx_err, err_loc)

    # Correct Bob's bits
    bob_corrected = bob_key
    for ii, idx in enumerate(idx_perm):
        bob_corrected[idx] = bob_perm[ii]

    idx_err = idx_perm[idx_err]  # Indices needing correction
    eve_information = len(idx_blocks)  # Information gained by Eve

    return bob_corrected, idx_err, eve_information


def information_reconciliation(
    alice_key: NDArray[np.int64],
    bob_key: NDArray[np.int64],
    qber_estimate: float,
    n_passes: int = 4,
    seed: int = None,
) -> Tuple[NDArray[np.int64], NDArray[np.int64]]:
    """
    Perform information reconciliation by performing the Cascade protocol
    `n_passes` times
    """

    rng = np.random.default_rng(seed)
    n = len(alice_key)

    # First block-size is inversely proportional to QBER
    block_size = int(1 / qber_estimate)
    eve_information = 0  # Accumulate additional information gained by Eve
    for _ in range(n_passes):
        perm = np.arange(n)
        rng.shuffle(perm)

        bob_key, _, eve_bits_acc = cascade_protocol(
            alice_key, bob_key, block_size, perm
        )
        eve_information += eve_bits_acc

        block_size *= 2  # Increase block size for next iteration

    return bob_key, eve_information

In [ ]:
def information_reconciliation_experiment(num_try, losses, fidelities, backend=hw_backend, bit_num: int = 20, seed: int = 123, sampler = None):
    # Setup
    rng = get_rn_generator(seed)
    if sampler is None:
        if service is not None:
            sampler = Sampler(mode=backend)
        else:
            sampler = StatevectorSampler()

    # Alice selects random bits and bases
    abits = sample_bits(bit_num, rng)
    abase = sample_bits(bit_num, rng)
    
    # Part 1: Alice -> Eve
    qc_eve = QuantumCircuit(bit_num, bit_num)
    
    # Alice prepares qubits
    encode_bb84_states(qc_eve, abits, abase)
    qc_eve.barrier()
    
    # Eve measures in her chosen bases
    ebase = sample_bits(bit_num, rng)
    measure_bb84(qc_eve, ebase)

    # On hardware
    # Transpile for the backend
    pm = generate_preset_pass_manager(
        target=backend.target,
        optimization_level=3
    )
    qc_eve_isa = pm.run(qc_eve)
    
    # Execute Eve's interception on hardware
    job_eve = sampler.run([qc_eve_isa], shots=1)
    print("Eve Job ID:", job_eve.job_id())
    result_eve = job_eve.result()
    counts_eve, countsint_eve = extract_counts(result_eve)
    # Post-processing
    ebits = counts_to_bbits(counts_eve, bit_num)
    print("Eve's bits:\n", ebits)
    
    # Part 2: Eve -> Bob
    # Eve construct guess states to send on to Bob
    qc_bob = QuantumCircuit(bit_num, bit_num)
    # Eve encodes using her measured bits in her chosen bases
    encode_bb84_states(qc_bob, ebits, ebase)
    qc_bob.barrier()
    
    # Bob measures in his chosen bases
    bbase = sample_bits(bit_num, rng)
    measure_bb84(qc_bob, bbase)
        
    # Transpile for the backend
    qc_bob_isa = pm.run(qc_bob)
    # Execute Bob's measurement on hardware
    job_bob = sampler.run([qc_bob_isa], shots=1)
    print("Bob Job ID:", job_bob.job_id())
    result_bob = job_bob.result()

    # Post-process Bob's measurement
    counts_bob, countsint_bob = extract_counts(result_bob)
    bbits = counts_to_bbits(counts_bob, bit_num)

    # Bits where Alice and Bob chose the same basis
    agoodbits, bgoodbits = sift_key(abits, abase, bbase, bbits)

    qber_estimate = bit_error_rate(
        input_bits=agoodbits[0:5], output_bits=bgoodbits[0:5]
    )  # Estimate QBER using subset of remaining message

    print(f"Estimated QBER: {qber_estimate} \n")

    actual_qber = bit_error_rate(
        input_bits=agoodbits, output_bits=bgoodbits
    )

    print(f"Actual QBER: {actual_qber} \n")

    bcorrectedbits, _ = information_reconciliation(
        alice_key=agoodbits,
        bob_key=bgoodbits,
        qber_estimate=qber_estimate,
        n_passes=4,
        seed=seed,
    )

    corrected_qber = bit_error_rate(
        input_bits=agoodbits, output_bits=bcorrectedbits
    )

    print(f"Post-information reconciliation QBER: {corrected_qber} \n")


In [ ]:
# Entry point
trial_num = 5

for backend_index, (backend_obj, backend_name) in enumerate(zip([sim_backend, noisy_sim_backend], ["sim", "noisy_sim"])):
    fidelities = []
    losses = []
    record = []
    for trial in range(trial_num):
        print("Try ", trial)
        record.append(information_reconciliation_experiment(trial, losses, fidelities, backend=backend_obj))
    with open(f"experiment4_{backend_name}.pkl", "wb") as fid:
        pickle.dump(record, fid)

## 6. Experiment 5: Privacy Amplification
- Hashing SHA
- Compress key
- Final key could be shorter but secure b/c Eve information decreases 

## References:
https://quantum.cloud.ibm.com/learning/en/modules/computer-science/quantum-key-distribution

In [ ]:
def binary_entropy(p: float) -> float:
    """
    Evaluates the binary entropy function
    """
    return (-p * np.log2(p) - (1 - p) * np.log2(1 - p))

def privacy_amplification(
    reconciled_key: NDArray[np.int64],
    eve_information: int,
    qber: float,
    rng: np.random.Generator = np.random.default_rng(),
) -> NDArray[np.int64]:
    """
    Privacy amplification using a random binary matrix as the hash matrix
    """

    # Key length after privacy amplification
    key_length = np.floor(
        len(reconciled_key) * (1 - binary_entropy(qber)) - eve_information
    )
    if key_length <= 0:
        return None

    m = int(key_length)
    n = len(reconciled_key)

    hash_matrix = rng.integers(0, 2, size=(m, n), dtype=np.uint16)

    return (hash_matrix @ reconciled_key) % 2



In [ ]:

def privacy_amplification_experiment():
    # Setup
    bit_num = 20
    rng = get_rn_generator(123)
    # Alice selects random bits and bases
    abits = sample_bits(bit_num, rng)
    abase = sample_bits(bit_num, rng)
    
    # Part 1: Alice -> Eve
    qc_eve = QuantumCircuit(bit_num, bit_num)
    
    # Alice prepares qubits
    encode_bb84_states(qc_eve, abits, abase)
    qc_eve.barrier()
    
    # Eve measures in her chosen bases
    ebase = sample_bits(bit_num, rng)
    measure_bb84(qc_eve, ebase)

    # On hardware
    # Transpile for the backend
    pm = generate_preset_pass_manager(
        target=backend.target,
        optimization_level=1
    )
    qc_eve_isa = pm.run(qc_eve)
    
    # Execute Eve's interception on hardware
    sampler = Sampler(mode=backend)
    job_eve = sampler.run([qc_eve_isa], shots=1)
    print("Eve Job ID:", job_eve.job_id())
    result_eve = job_eve.result()
    counts_eve, countsint_eve = extract_counts(result_eve)
    # Post-processing
    ebits = counts_to_bbits(counts_eve, bit_num)
    print("Eve's bits:\n", ebits)
    
    # Part 2: Eve -> Bob
    # Eve construct guess states to send on to Bob
    qc_bob = QuantumCircuit(bit_num, bit_num)
    # Eve encodes using her measured bits in her chosen bases
    encode_bb84_states(qc_bob, ebits, ebase)
    qc_bob.barrier()
    
    # Bob measures in his chosen bases
    bbase = sample_bits(bit_num, rng)
    measure_bb84(qc_bob, bbase)
        
    # Transpile for the backend
    qc_bob_isa = pm.run(qc_bob)
    # Execute Bob's measurement on hardware
    job_bob = sampler.run([qc_bob_isa], shots=1)
    print("Bob Job ID:", job_bob.job_id())
    result_bob = job_bob.result()

    # Post-process Bob's measurement
    counts_bob, countsint_bob = extract_counts(result_bob)
    bbits = counts_to_bbits(counts_bob, bit_num)

    # Bits where Alice and Bob chose the same basis
    agoodbits, bgoodbits = sift_key(abits, abase, bbase, bbits)

    qber_estimate = bit_error_rate(
        input_bits=agoodbits[0:5], output_bits=agoodbits[0:5]
    )  # Estimate QBER using subset of remaining message

    print(f"Estimated QBER: {qber_estimate} \n")

    actual_qber = bit_error_rate(
        input_bits=agoodbits, output_bits=bgoodbits
    )

    print(f"Actual QBER: {actual_qber} \n")

    bcorrectedbits, eve_information = information_reconciliation(
        alice_keyy=agoodbits,
        bob_key=bgoodbits,
        qber_estimate=qber_estimate,
        n_passes=4,
        seed=123,
    )

    corrected_qber = bit_error_rate(
        input_bits=agoodbits, output_bits=bcorrectedbits
    )

    print(f"Post-information reconciliation QBER: {corrected_qber} \n")

    # This will return None until we can lower the frequency at which Eve
    # intercepts the channel
    secret_key = privacy_amplification(
        reconciled_key=bcorrectedbits,
        eve_information=eve_information,
        qber=qber_estimate,
        rng=rng,
    )

    print(f"Final secret key: {secret_key}")

In [ ]:
privacy_amplification_experiment()